# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hanizakkk/flyrank_working-repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

> **TODO (you, not me):** I don't have access to the FlyRank research paper
> referenced here, so I can't honestly critique its findings without
> fabricating quotes or statistics I never saw. Pick two findings from that
> paper yourself and, for each, note: where the label came from, and whether
> the validation design (random vs. grouped/time-aware split) could have
> inflated the reported number. Use the same lens as section 2 below — that's
> a worked example of exactly this exercise on my own model.

In [1]:
# Leave this cell for whatever numbers you pull while answering section 1
# above, once you have the paper in front of you.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before/after, same model, same test items:** the honest (client-holdout)
split isn't just theoretically better — it produces a measurably different
number than a naive random split, on the exact same March->April dev data:

| Split | Precision@50 | Client overlap (train vs test) |
|---|---|---|
| Random row split | 0.96 | 45 clients |
| Client holdout split | **0.86** | 0 clients |

The random split lets 45 clients appear in both train and test, so the model
partly learns each client's baseline traffic level instead of a transferable
decline signal — a full 10-point inflation. All results reported anywhere
else in this capstone use the client-holdout number (0.86), never the
optimistic one.

In [2]:
import json
with open("../outputs/split_comparison.json") as f:
    split_comparison = json.load(f)
print(json.dumps(split_comparison, indent=2))

[
  {
    "base_rate": 0.5913276568905708,
    "client_overlap": 45,
    "precision_at_50": 0.96,
    "split": "random_row_split"
  },
  {
    "base_rate": 0.6620141722138716,
    "client_overlap": 0,
    "precision_at_50": 0.86,
    "split": "client_holdout_split"
  }
]


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Checklist run against the final 8-feature set** (see
`work/scripts/warehouse_utils.py::FEATURE_COLUMNS` and
`aggregate_month_features` / `aggregate_month_outcome`):

- Features come only from the decision month's aggregation query — the
  outcome month is never read in that function.
- The label (`future_decline`) is built from a separate query against the
  outcome month only, and is never included in the feature list.
- `client_hash_id` / `content_hash_id` are used only to join and group rows —
  never passed to a model as a feature.
- All reported results use the client-holdout split.

**Deliberate leak demonstration:** folding `future_decline` into its own
feature set and retraining pushes precision@50 from the honest 0.86 to a
perfect **1.000** — the model just reads its own answer back. This confirms
the audit process actually catches leakage when it's present, not just that
the honest features happen to look clean.

In [3]:
with open("../outputs/leakage_audit.json") as f:
    audit = json.load(f)
print(json.dumps(audit, indent=2))

{
  "checklist": {
    "features_only_from_decision_month": true,
    "grouped_split_used_for_reported_results": true,
    "ids_excluded_from_features": true,
    "no_label_or_sibling_columns_in_features": true
  },
  "leaky_precision_at_50": 1.0
}


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence, before:** "The model predicts which pages will decline in
search traffic."

**Rewritten, safe version:** "On a client-holdout validation split, the model's
top-50 ranked items were associated with an observed decline in the following
month's search impressions 86% of the time — a decision-support signal for
prioritizing manual refresh review, not a causal or algorithmic prediction of
what Google will do to any specific page.

In [4]:
print("Claim rewritten above (markdown cell) - no numeric result needed here.")

Claim rewritten above (markdown cell) - no numeric result needed here.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.